# Train Qwen3.5-4B for the GRC app

Fine-tunes Qwen3.5-4B with Unsloth (a LoRA add-on, 4-bit) on the app's AI jobs, then exports one GGUF file for Unsloth Studio. Your steps are in `docs/training/README.md`.

- **Data:** `train.jsonl` in Drive's `grc-training` folder, built on the Mac from real public data (MITRE ATT&CK, NVD, CISA KEV, VCDB, text2cypher, NIST and ISO catalogs). Company X is never in it.
- **Jobs:** `extraction` (find the links a document states), `graph_query` (write the graph query for a question) and, from round 2, `answer` (the chat answer with citations).
- **Time:** round 2's file (4,423 rows) is one pass of about 990 steps: about 6–7 h on a T4, roughly half on an L4, then the check and the export.
- **Disconnects:** reconnect and choose **Runtime → Run all**. It resumes from the last checkpoint (about every 20 min) and skips finished parts.
- **Writes to Drive:** `checkpoints/`, `adapter/`, `validation.json` and the `.gguf` file.
- **Base-only mode** (`BASE_ONLY = True`): exports the untrained model as `qwen3.5-4b-base.Q4_K_M.gguf` for the before-test (about 30 min) and trains nothing.

No Hugging Face token is needed; ignore Colab's warning about it.

Cells: 1. this note · 2. Settings · 3. Install · 4. Data · 5. Model · 6. Format · 7. Train · 8. Validation check · 9. Export · 10. Next steps.

## 2. Settings

The defaults fit a free T4. Before you run, check `BASE_ONLY` and `EPOCHS`.

In [ ]:
# ---- Mode ---------------------------------------------------------------------------------------
BASE_ONLY = False  # True: only export the untrained model for the before-test

# ---- Files in Google Drive ----------------------------------------------------------------------
DRIVE_FOLDER = "/content/drive/MyDrive/grc-training"
DATA_FILE = "train.jsonl"
JOBS = None  # None trains every job in the file, or a list such as ["extraction"]

# ---- Model --------------------------------------------------------------------------------------
BASE_MODEL = "unsloth/Qwen3.5-4B"
LOAD_IN_4BIT = True  # keep True: a T4 trains in float32, so 16-bit weights don't fit
MAX_SEQ_LENGTH = 4096  # tokens per row; a longer row is skipped, never cut

# ---- Training (Unsloth's Qwen3.5 recipe) --------------------------------------------------------
EPOCHS = 1  # passes over the training rows
LORA_RANK = 16
LORA_ALPHA = 16
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
BATCH_SIZE = 1
GRAD_ACCUM = 4  # one step = BATCH_SIZE x GRAD_ACCUM = 4 rows
LEARNING_RATE = 2e-4
WARMUP_STEPS = 10
WEIGHT_DECAY = 0.001
LR_SCHEDULER = "linear"
OPTIMIZER = "adamw_8bit"
SEED = 42

# ---- Checkpoints and logs -----------------------------------------------------------------------
SAVE_STEPS = 50  # about 20 min on a T4; the last 2 are kept
LOGGING_STEPS = 10

# ---- Validation check and export ----------------------------------------------------------------
VALIDATION_ROWS_PER_JOB = 50
QUANT = "q4_k_m"  # keep the same for the base and trained files, so the two tests compare like for like

# ---- Checks on the settings (nothing to change below) -------------------------------------------
import hashlib
import json

JOB_NAMES = ("extraction", "mapping", "graph_query", "answer")
JSON_JOBS = ("extraction", "mapping", "graph_query")
FOUR_BIT_QUANTS = ("q4_k_m", "q4_k_s", "q4_0", "q4_1")


class StopHere(Exception):
    # Stops "Run all" with a plain message and no traceback.
    def _render_traceback_(self):
        return ["", "STOPPED: " + str(self), ""]


def _need(ok, message):
    if not ok:
        raise StopHere(message)


def _is_int(value):
    return isinstance(value, int) and not isinstance(value, bool)


def _is_number(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool)


def run_tag(identity):
    # A short name for one training: the same file and settings always give the same tag.
    return hashlib.sha256(json.dumps(identity, sort_keys=True).encode("utf-8")).hexdigest()[:8]


_need(isinstance(BASE_ONLY, bool), "BASE_ONLY must be True or False.")
_need(
    isinstance(DRIVE_FOLDER, str) and DRIVE_FOLDER.startswith("/content/drive/"),
    "DRIVE_FOLDER must be a folder under /content/drive/ (for example /content/drive/MyDrive/grc-training).",
)
_need(
    isinstance(DATA_FILE, str) and DATA_FILE.strip() != "" and "/" not in DATA_FILE,
    "DATA_FILE must be a file name such as train.jsonl, without a folder.",
)
_need(
    JOBS is None
    or (
        isinstance(JOBS, (list, tuple))
        and len(JOBS) > 0
        and all(job in JOB_NAMES for job in JOBS)
        and len(set(JOBS)) == len(JOBS)
    ),
    f"JOBS must be None, or a list of different job names from {list(JOB_NAMES)}.",
)
_need(isinstance(BASE_MODEL, str) and BASE_MODEL.strip() != "", "BASE_MODEL must be a model ID.")
_need(isinstance(LOAD_IN_4BIT, bool), "LOAD_IN_4BIT must be True or False.")
for _name in (
    "MAX_SEQ_LENGTH",
    "LORA_RANK",
    "LORA_ALPHA",
    "BATCH_SIZE",
    "GRAD_ACCUM",
    "SAVE_STEPS",
    "LOGGING_STEPS",
    "VALIDATION_ROWS_PER_JOB",
):
    _need(_is_int(globals()[_name]) and globals()[_name] > 0, f"{_name} must be a whole number above 0.")
for _name in ("WARMUP_STEPS", "SEED"):
    _need(_is_int(globals()[_name]) and globals()[_name] >= 0, f"{_name} must be a whole number, 0 or more.")
_need(_is_number(EPOCHS) and EPOCHS > 0, "EPOCHS must be a number above 0.")
_need(_is_number(LEARNING_RATE) and LEARNING_RATE > 0, "LEARNING_RATE must be a number above 0.")
_need(_is_number(WEIGHT_DECAY) and WEIGHT_DECAY >= 0, "WEIGHT_DECAY must be a number, 0 or more.")
_need(
    isinstance(LORA_TARGET_MODULES, (list, tuple))
    and len(LORA_TARGET_MODULES) > 0
    and all(isinstance(module, str) and module != "" for module in LORA_TARGET_MODULES),
    "LORA_TARGET_MODULES must be a list of module names.",
)
_need(isinstance(LR_SCHEDULER, str) and LR_SCHEDULER != "", "LR_SCHEDULER must be a scheduler name such as linear.")
_need(isinstance(OPTIMIZER, str) and OPTIMIZER != "", "OPTIMIZER must be an optimiser name such as adamw_8bit.")
_need(QUANT in FOUR_BIT_QUANTS, f"QUANT must be one of {list(FOUR_BIT_QUANTS)}.")
print("The settings are fine.")

## 3. Install

Unsloth and its libraries at exact versions. Takes a few minutes, and runs again after every disconnect.

In [ ]:
import importlib
import shutil
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

# A GPU first: without one there is nothing to train on.
_gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True) if shutil.which("nvidia-smi") else None
_need(
    _gpu is not None and _gpu.returncode == 0 and _gpu.stdout.strip() != "",
    "This runtime has no GPU. Choose Runtime -> Change runtime type -> T4 GPU, then Runtime -> Run all.",
)
print("GPU:", _gpu.stdout.strip())

# The newest versions Unsloth 2026.10.1 accepts; torch stays at the build Colab ships, so nothing big
# is downloaded after a disconnect. Their dependencies are limited to releases before EXCLUDE_NEWER.
PINNED = {
    "torch": "2.11.0",
    "torchvision": "0.26.0",
    "unsloth": "2026.10.1",
    "unsloth_zoo": "2026.10.1",
    "transformers": "5.17.0",
    "trl": "1.13.0",
    "peft": "0.21.2",
    "accelerate": "1.15.0",
    "datasets": "4.8.5",
    "bitsandbytes": "0.50.2",
    "xformers": "0.0.35",
    "torchao": "0.18.0",
}
EXCLUDE_NEWER = "2026-10-07T00:00:00Z"
PIN_ARGS = " ".join(f"{name}=={pinned}" for name, pinned in PINNED.items())

!pip install -q "uv==0.12.23"
!uv pip install -q --system --python {sys.executable} --exclude-newer {EXCLUDE_NEWER} {PIN_ARGS}
# Prebuilt kernels for Qwen3.5's linear-attention layers; falls back to plain torch when none fits.
!unsloth install-kernels

importlib.invalidate_caches()
for _name, _wanted in PINNED.items():
    try:
        _got = version(_name)
    except PackageNotFoundError:
        _got = None
    _need(
        _got is not None and _got.split("+")[0] == _wanted,
        f"{_name} is {_got}, not {_wanted}. Choose Runtime -> Disconnect and delete runtime, then Runtime -> Run all.",
    )
    print(f"{_name} {_got}")

## 4. Data

Mounts Google Drive (allow access when Colab asks), checks every row of `train.jsonl` and prints the rows per job. It stops on the first bad row, or if Drive holds another training's checkpoints. In base-only mode it only checks that the folder is there.

In [ ]:
from collections import Counter
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

DRIVE = Path(DRIVE_FOLDER)
TRAIN_FILE = DRIVE / DATA_FILE
CHECKPOINT_DIR = DRIVE / "checkpoints"
ADAPTER_DIR = DRIVE / "adapter"
VALIDATION_FILE = DRIVE / "validation.json"

if BASE_ONLY:
    _need(
        DRIVE.is_dir(),
        f"Google Drive has no folder {DRIVE_FOLDER}. Make a folder named grc-training in My Drive (nothing to upload "
        "for the base file), then Runtime -> Run all.",
    )
    print(f"Base-only mode: no training file is read. The base file goes to {DRIVE_FOLDER}.")
else:
    _need(
        DRIVE.is_dir(),
        f"Google Drive has no folder {DRIVE_FOLDER}. Make a folder named grc-training in My Drive, upload {DATA_FILE} "
        "into it, then Runtime -> Run all.",
    )
    _need(
        TRAIN_FILE.is_file(),
        f"There is no {DATA_FILE} in {DRIVE_FOLDER}. Upload the training file from the Mac, then Runtime -> Run all.",
    )

    ROLES = ("system", "user", "assistant")
    SPLITS = ("train", "validation")

    def row_problem(row):
        # What is wrong with one training row, or None when it is fine.
        if not isinstance(row, dict):
            return "it is not a JSON object"
        if set(row) != {"job", "messages", "split", "seed"}:
            return "it must have exactly the four columns job, messages, split and seed"
        if row["job"] not in JOB_NAMES:
            return f"job must be one of {', '.join(JOB_NAMES)}"
        messages = row["messages"]
        if not isinstance(messages, list) or len(messages) != len(ROLES):
            return "messages must be three messages: system, user and assistant"
        for message, role in zip(messages, ROLES):
            if not isinstance(message, dict) or set(message) != {"role", "content"} or message["role"] != role:
                return "messages must be system, user and assistant, in that order, each with only a role and a content"
            if not isinstance(message["content"], str) or message["content"].strip() == "":
                return f"the {role} message's content must be text, not empty"
        if row["job"] in JSON_JOBS:
            try:
                json.loads(messages[2]["content"])
            except ValueError:
                return f"the assistant answer must be JSON for the {row['job']} job"
        if row["split"] not in SPLITS:
            return "split must be train or validation"
        if not _is_int(row["seed"]) or row["seed"] < 0:
            return "seed must be a whole number"
        return None

    def read_rows(path):
        # The file's SHA-256 and its rows, each one checked.
        raw = path.read_bytes()
        try:
            text = raw.decode("utf-8")
        except UnicodeDecodeError:
            raise StopHere(f"{path.name} is not UTF-8 text. Build it again on the Mac.") from None
        lines = text.split("\n")
        while lines and lines[-1].strip() == "":
            lines.pop()
        _need(lines, f"{path.name} is empty. Build it again on the Mac.")
        rows = []
        for number, line in enumerate(lines, start=1):
            _need(line.strip() != "", f"Line {number} of {path.name} is empty. Build it again on the Mac.")
            try:
                row = json.loads(line)
            except ValueError:
                raise StopHere(f"Line {number} of {path.name} is not valid JSON. Build it again on the Mac.") from None
            problem = row_problem(row)
            _need(problem is None, f"Line {number} of {path.name} is not a training row: {problem}. Build it again on the Mac.")
            rows.append(row)
        return hashlib.sha256(raw).hexdigest(), rows

    def saved_identity(folder):
        # The run.json a training left in `folder`: None when there is none, {} when it can't be read.
        path = folder / "run.json"
        if not path.is_file():
            return None
        try:
            return json.loads(path.read_text())
        except ValueError:
            return {}

    TRAIN_FILE_SHA256, ALL_ROWS = read_rows(TRAIN_FILE)
    ROWS = [row for row in ALL_ROWS if JOBS is None or row["job"] in JOBS]
    _missing = [job for job in (JOBS or []) if not any(row["job"] == job for row in ROWS)]
    _need(
        not _missing,
        f"{DATA_FILE} has no rows for {', '.join(_missing)}. Change JOBS in Settings, or build the file with those jobs.",
    )
    _need(any(row["split"] == "train" for row in ROWS), "The chosen jobs have no train rows.")
    TRAINED_JOBS = [job for job in JOB_NAMES if any(row["job"] == job for row in ROWS)]
    _counts = Counter((row["job"], row["split"]) for row in ROWS)

    print(f"{DATA_FILE}: {len(ALL_ROWS)} rows, all fine. {len(ROWS)} rows for the chosen jobs:")
    print(f"  {'job':<12} {'train':>7} {'validation':>11}")
    for job in TRAINED_JOBS:
        print(f"  {job:<12} {_counts[(job, 'train')]:>7} {_counts[(job, 'validation')]:>11}")

    # The file and every setting that changes the result. Saved as run.json with the checkpoints and the
    # adapter, so a training never resumes from another one's files.
    IDENTITY = {
        "trainFileSha256": TRAIN_FILE_SHA256,
        "jobs": TRAINED_JOBS,
        "baseModel": BASE_MODEL,
        "loadIn4bit": LOAD_IN_4BIT,
        "maxSeqLength": MAX_SEQ_LENGTH,
        "epochs": EPOCHS,
        "loraRank": LORA_RANK,
        "loraAlpha": LORA_ALPHA,
        "loraTargetModules": list(LORA_TARGET_MODULES),
        "batchSize": BATCH_SIZE,
        "gradAccum": GRAD_ACCUM,
        "learningRate": LEARNING_RATE,
        "warmupSteps": WARMUP_STEPS,
        "weightDecay": WEIGHT_DECAY,
        "lrScheduler": LR_SCHEDULER,
        "optimizer": OPTIMIZER,
        "seed": SEED,
    }
    RUN_TAG = run_tag(IDENTITY)

    _other = "{} holds another training's files (a different {} or other settings). Delete or rename the checkpoints and adapter folders in {}, then Runtime -> Run all."
    _saved = saved_identity(CHECKPOINT_DIR)
    _need(_saved is None or _saved == IDENTITY, _other.format(CHECKPOINT_DIR, DATA_FILE, DRIVE_FOLDER))
    _need(
        _saved is not None or not any(CHECKPOINT_DIR.glob("checkpoint-*")),
        _other.format(CHECKPOINT_DIR, DATA_FILE, DRIVE_FOLDER),
    )
    _saved = saved_identity(ADAPTER_DIR)
    _need(_saved is None or _saved == IDENTITY, _other.format(ADAPTER_DIR, DATA_FILE, DRIVE_FOLDER))
    print(f"This training's tag: {RUN_TAG}")

## 5. Model

Loads Qwen3.5-4B in 4-bit and adds the LoRA add-on.

In [ ]:
if BASE_ONLY:
    print("Base-only mode: skipped (the Export cell loads the untrained model itself).")
else:
    from unsloth import FastLanguageModel  # import Unsloth first, so it can patch transformers and TRL

    import torch

    model, processor = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=LOAD_IN_4BIT,
        full_finetuning=False,
    )
    # Qwen3.5 comes with a processor (it also reads images); training uses its text tokenizer.
    tokenizer = getattr(processor, "tokenizer", processor)

    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_RANK,
        target_modules=list(LORA_TARGET_MODULES),
        lora_alpha=LORA_ALPHA,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
        max_seq_length=MAX_SEQ_LENGTH,
    )
    TRAINED_TAG = None  # set once the trained weights are in `model`
    print(f"Loaded. GPU memory in use: {torch.cuda.max_memory_reserved() / 2**30:.1f} GB.")

## 6. Format

Writes each row in the model's chat template with thinking off, exactly as the app asks. A row longer than `MAX_SEQ_LENGTH` is skipped and counted. One row is printed so you can see it.

In [ ]:
if BASE_ONLY:
    print("Base-only mode: skipped (no training rows are read).")
else:
    from datasets import Dataset

    # The chat template sits on the tokenizer; the processor has the same one.
    CHAT = tokenizer if getattr(tokenizer, "chat_template", None) else processor

    def chat_text(messages, prompt_only=False):
        # The row in the chat template, thinking off. prompt_only: just what the model is given when the app asks.
        return CHAT.apply_chat_template(
            messages[:2] if prompt_only else messages,
            tokenize=False,
            add_generation_prompt=prompt_only,
            enable_thinking=False,
        )

    def token_count(text):
        return len(tokenizer(text, add_special_tokens=False)["input_ids"])

    train_texts = []
    validation_rows = []
    _skipped = Counter()
    for row in ROWS:
        text = chat_text(row["messages"])
        prompt = chat_text(row["messages"], prompt_only=True)
        answer = row["messages"][2]["content"]
        # What is trained must be exactly the app's prompt followed by the answer.
        _need(
            isinstance(text, str)
            and isinstance(prompt, str)
            and text.startswith(prompt)
            and text[len(prompt) :].startswith(answer.strip()),
            "The chat template doesn't put the answer right after the app's prompt, so training would not match how "
            "the app asks. Nothing was trained.",
        )
        # +1 for the end token the trainer may add.
        if token_count(text) + 1 > MAX_SEQ_LENGTH:
            _skipped[(row["job"], row["split"])] += 1
            continue
        if row["split"] == "train":
            train_texts.append(text)
        else:
            validation_rows.append(
                {"job": row["job"], "user": row["messages"][1]["content"], "prompt": prompt, "answer": answer, "text": text}
            )

    if _skipped:
        print(f"Skipped as longer than MAX_SEQ_LENGTH ({MAX_SEQ_LENGTH} tokens):")
        for (job, split), count in sorted(_skipped.items()):
            print(f"  {job} {split}: {count}")
    else:
        print(f"No row is longer than MAX_SEQ_LENGTH ({MAX_SEQ_LENGTH} tokens).")
    _need(train_texts, "Every train row is longer than MAX_SEQ_LENGTH, so there is nothing to train on.")
    print(f"{len(train_texts)} train rows and {len(validation_rows)} validation rows are used.")

    TRAIN_DATASET = Dataset.from_dict({"text": train_texts})
    VALIDATION_DATASET = Dataset.from_dict({"text": [row["text"] for row in validation_rows]}) if validation_rows else None

    print()
    print("One training row as the model sees it:")
    print(train_texts[0])

## 7. Train

Learns the assistant answers only; the system and user parts are context. The loss is printed every 10 steps and should trend down.

- Checkpoints go to `grc-training/checkpoints` (the last 2 kept). A rerun resumes from the newest one.
- When training ends, the adapter is saved to `grc-training/adapter`. If it is already there, it is loaded instead of training again.

In [ ]:
if BASE_ONLY:
    print("Base-only mode: skipped (nothing is trained).")
else:
    import shutil

    from peft import load_peft_weights, set_peft_model_state_dict
    from trl import SFTConfig, SFTTrainer
    from unsloth.chat_templates import train_on_responses_only

    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=TRAIN_DATASET,
        eval_dataset=VALIDATION_DATASET,
        args=SFTConfig(
            output_dir=str(CHECKPOINT_DIR),
            dataset_text_field="text",
            max_length=MAX_SEQ_LENGTH,
            per_device_train_batch_size=BATCH_SIZE,
            per_device_eval_batch_size=BATCH_SIZE,
            gradient_accumulation_steps=GRAD_ACCUM,
            num_train_epochs=EPOCHS,
            learning_rate=LEARNING_RATE,
            warmup_steps=WARMUP_STEPS,
            weight_decay=WEIGHT_DECAY,
            lr_scheduler_type=LR_SCHEDULER,
            optim=OPTIMIZER,
            logging_steps=LOGGING_STEPS,
            save_strategy="steps",
            save_steps=SAVE_STEPS,
            save_total_limit=2,
            eval_strategy="no",
            prediction_loss_only=True,
            seed=SEED,
            data_seed=SEED,
            dataset_num_proc=1,
            report_to="none",
        ),
    )
    # Only the answers are learnt: the template's user and assistant markers show where they are.
    trainer = train_on_responses_only(
        trainer,
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n",
    )
    _first = trainer.train_dataset[0]
    print("The part of one training row that is learnt:")
    print(tokenizer.decode([token for token, label in zip(_first["input_ids"], _first["labels"]) if label != -100]))
    print()

    def step_of(folder):
        try:
            return int(folder.name.rsplit("-", 1)[-1])
        except ValueError:
            return -1

    def checkpoint_state(folder):
        # The trainer's saved state, or None when the checkpoint was cut off while saving.
        needed = ("trainer_state.json", "adapter_model.safetensors", "optimizer.pt", "scheduler.pt")
        if not all((folder / name).is_file() for name in needed):
            return None
        try:
            return json.loads((folder / "trainer_state.json").read_text())
        except ValueError:
            return None

    def load_trained_weights(folder):
        # Puts the LoRA weights saved in `folder` into the model.
        weights = load_peft_weights(str(folder), device="cuda")
        result = set_peft_model_state_dict(model, weights)
        _need(
            bool(weights) and not result.unexpected_keys,
            f"The weights in {folder} don't fit this model. Delete the checkpoints and adapter folders in "
            f"{DRIVE_FOLDER}, then Runtime -> Run all to train again.",
        )

    def save_adapter():
        # run.json goes last: an adapter without it isn't finished.
        ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
        (ADAPTER_DIR / "run.json").unlink(missing_ok=True)
        model.save_pretrained(str(ADAPTER_DIR))
        processor.save_pretrained(str(ADAPTER_DIR))
        (ADAPTER_DIR / "run.json").write_text(json.dumps(IDENTITY, indent=2))
        print(f"The LoRA adapter is saved in {ADAPTER_DIR}.")

    if saved_identity(ADAPTER_DIR) == IDENTITY:
        print(f"This training ({RUN_TAG}) finished in an earlier session: loading its saved adapter.")
        load_trained_weights(ADAPTER_DIR)
    else:
        CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
        if saved_identity(CHECKPOINT_DIR) is None:
            (CHECKPOINT_DIR / "run.json").write_text(json.dumps(IDENTITY, indent=2))
        # The newest complete checkpoint; half-saved ones are removed.
        _last, _last_state = None, None
        for _folder in sorted(CHECKPOINT_DIR.glob("checkpoint-*"), key=step_of, reverse=True):
            _state = checkpoint_state(_folder)
            if _state is not None:
                _last, _last_state = _folder, _state
                break
            print(f"Removing {_folder.name}: it was cut off while saving.")
            shutil.rmtree(_folder, ignore_errors=True)
        if _last is not None and _last_state.get("global_step", 0) >= _last_state.get("max_steps", float("inf")):
            print(f"Training finished at step {_last_state['global_step']} in an earlier session: loading that checkpoint.")
            load_trained_weights(_last)
        else:
            if _last is None:
                print("No checkpoint yet: training starts from the beginning.")
            else:
                print(f"Resuming from the checkpoint at step {_last_state['global_step']} of {_last_state['max_steps']}.")
            _stats = trainer.train(resume_from_checkpoint=str(_last) if _last is not None else None)
            print(f"Training done: {_stats.global_step} steps, training loss {_stats.training_loss:.4f}.")
            print(f"Peak GPU memory: {torch.cuda.max_memory_reserved() / 2**30:.1f} GB.")
        save_adapter()
    TRAINED_TAG = RUN_TAG

## 8. Validation check

The validation loss, then the model answers up to `VALIDATION_ROWS_PER_JOB` validation rows per job (thinking off, no sampling). For each job it prints the share of answers that parse as JSON and the share equal to the expected answer, and saves the table to Drive as `validation.json`. It takes a while: each answer is written token by token.

In [ ]:
if BASE_ONLY:
    print("Base-only mode: skipped (there is no trained model to check).")
else:
    import random
    import re

    _need(
        globals().get("TRAINED_TAG") is not None and globals().get("TRAINED_TAG") == globals().get("RUN_TAG"),
        "The trained model isn't loaded in this session. Choose Runtime -> Run all.",
    )

    def without_thinking(text):
        # The answer without any thinking text, as the app drops it.
        while "<think>" in text:
            start = text.index("<think>")
            end = text.find("</think>", start)
            text = text[:start] + ("" if end == -1 else text[end + len("</think>") :])
        if "</think>" in text:
            text = text.split("</think>")[-1]
        return text.strip()

    # For the answer job: citations read as the app reads them (packages/api/src/chat/citations.ts),
    # such as [1], [2][3], [1, 3], [2-4] or [Source 2].
    _BRACKET = re.compile(r"\[([^\[\]\n]{1,80})\]")
    _SEPARATOR = re.compile(r"\s*(?:,|;|&|\band\b)\s*", re.IGNORECASE)
    _ITEM = re.compile(r"(?:sources?\s*)?([0-9]{1,6})(?:\s*[-–—]\s*(?:sources?\s*)?([0-9]{1,6}))?", re.IGNORECASE)
    _MAX_RANGE = 50

    def _bracket_numbers(inner):
        numbers = []
        for part in _SEPARATOR.split(inner.strip()):
            match = _ITEM.fullmatch(part.strip())
            if match is None:
                return None
            low = int(match.group(1))
            high = low if match.group(2) is None else int(match.group(2))
            if high < low or high - low > _MAX_RANGE:
                numbers += [low, high]
            else:
                numbers += list(range(low, high + 1))
        return numbers

    def cited_numbers(answer):
        cited = set()
        for match in _BRACKET.finditer(answer):
            numbers = _bracket_numbers(match.group(1))
            if numbers is not None:
                cited.update(numbers)
        return cited

    def given_numbers(user_text):
        # The source numbers in the context: each record line and passage starts with [n].
        return {int(number) for number in re.findall(r"(?m)^\[([0-9]+)\] ", user_text)}

    def as_json(text):
        try:
            return True, json.loads(text)
        except ValueError:
            return False, None

    def write_answer(prompt, expected):
        inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to("cuda")
        limit = min(2 * token_count(expected) + 64, MAX_SEQ_LENGTH)
        with torch.no_grad():
            output = model.generate(**inputs, max_new_tokens=limit, do_sample=False, use_cache=True)
        written = output[0][inputs["input_ids"].shape[1] :]
        return without_thinking(tokenizer.decode(written, skip_special_tokens=True))

    def show(report):
        loss = report["validationLoss"]
        print("Validation loss (lower is better):", "no validation rows" if loss is None else f"{loss:.4f}")
        for job in TRAINED_JOBS:
            result = report["jobs"].get(job)
            if not result or result["rows"] == 0:
                print(f"{job}: no validation rows")
                continue
            rows = result["rows"]
            print(f"{job}: {rows} validation rows answered")
            for label, key in result["measures"]:
                print(f"  {label:<44} {result[key]:>4}/{rows}  {100 * result[key] / rows:5.1f}%")

    _saved_report = None
    if VALIDATION_FILE.is_file():
        try:
            _saved_report = json.loads(VALIDATION_FILE.read_text())
        except ValueError:
            _saved_report = None

    if isinstance(_saved_report, dict) and _saved_report.get("run") == RUN_TAG:
        print("The validation check already ran for this training. Its table:")
        show(_saved_report)
    else:
        report = {"run": RUN_TAG, "validationLoss": None, "jobs": {}}
        if trainer.eval_dataset is not None and len(trainer.eval_dataset) > 0:
            FastLanguageModel.for_training(model)
            report["validationLoss"] = trainer.evaluate()["eval_loss"]
        FastLanguageModel.for_inference(model)
        picker = random.Random(SEED)
        for job in TRAINED_JOBS:
            rows = [row for row in validation_rows if row["job"] == job]
            chosen = [rows[i] for i in sorted(picker.sample(range(len(rows)), min(VALIDATION_ROWS_PER_JOB, len(rows))))]
            first = second = 0
            for done, row in enumerate(chosen, start=1):
                written = write_answer(row["prompt"], row["answer"])
                if job in JSON_JOBS:
                    parses, value = as_json(written)
                    first += parses
                    second += parses and value == json.loads(row["answer"])
                else:
                    first += cited_numbers(written) <= given_numbers(row["user"])
                    second += written == row["answer"].strip()
                if done % 10 == 0 or done == len(chosen):
                    print(f"  {job}: {done}/{len(chosen)} answered")
            if job in JSON_JOBS:
                measures = [["answers that parse as JSON", "parsesAsJson"], ["answers equal to the expected answer", "equalsExpected"]]
                report["jobs"][job] = {"rows": len(chosen), "parsesAsJson": int(first), "equalsExpected": int(second), "measures": measures}
            else:
                measures = [["answers citing only the given sources", "citesOnlyGivenSources"], ["answers matching exactly", "matchesExactly"]]
                report["jobs"][job] = {"rows": len(chosen), "citesOnlyGivenSources": int(first), "matchesExactly": int(second), "measures": measures}
        VALIDATION_FILE.write_text(json.dumps(report, indent=2))
        print()
        show(report)
        print(f"Saved to {VALIDATION_FILE}.")

## 9. Export

Merges the adapter into the model and saves one `QUANT` GGUF file to Drive (built on Colab's disk, then copied). In base-only mode it exports the untrained model the same way, as `qwen3.5-4b-base.<QUANT>.gguf`.

After a disconnect you can run just Settings (2), Install (3) and this cell: it reloads the saved adapter from Drive.

In [ ]:
import gc
import os
import shutil
from pathlib import Path

if "PINNED" not in globals() or "QUANT" not in globals():
    raise SystemExit("Run the Settings cell (2) and the Install cell (3) first, then this cell.")

import torch
from google.colab import drive
from unsloth import FastLanguageModel

drive.mount("/content/drive")
DRIVE = Path(DRIVE_FOLDER)
ADAPTER_DIR = DRIVE / "adapter"


def export_gguf(export_model, export_tokenizer, gguf_name):
    # Builds the GGUF on Colab's own disk, then copies that one file to Drive as gguf_name.
    target = DRIVE / gguf_name
    work = Path("/content/grc-export/qwen3.5-4b-grc")
    shutil.rmtree(work.parent, ignore_errors=True)
    result = export_model.save_pretrained_gguf(str(work), export_tokenizer, quantization_method=QUANT)
    files = result.get("gguf_files") if isinstance(result, dict) else None
    if not files:
        files = [str(path) for path in Path(f"{work}_gguf").glob("*.gguf")]
    chosen = [f for f in files if QUANT.upper() in Path(f).name.upper() and "MMPROJ" not in Path(f).name.upper()]
    _need(len(chosen) == 1, f"Expected one {QUANT.upper()} GGUF file from the export, found {[Path(f).name for f in files]}.")
    # Copied under another name first, so a copy cut off by a disconnect never looks finished.
    partial = DRIVE / f"{gguf_name}.partial"
    shutil.copyfile(chosen[0], partial)
    os.replace(partial, target)
    print(f"Exported: {gguf_name}, {target.stat().st_size / 2**30:.2f} GB, in {DRIVE_FOLDER}.")


def export_base(gguf_name):
    # The untrained model for the before-test. Any trained model is let go first: the T4 can't hold two.
    for name in ("trainer", "model", "TRAINED_TAG"):
        globals().pop(name, None)
    gc.collect()
    torch.cuda.empty_cache()
    print(f"Loading the untrained {BASE_MODEL}.")
    base_model, base_processor = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=LOAD_IN_4BIT,
        full_finetuning=False,
    )
    base_tokenizer = getattr(base_processor, "tokenizer", base_processor)
    # A fresh LoRA add-on starts at zero, so it changes no weight, but it makes the export take the same
    # route as the trained model (4-bit load, merge, quantise). Without it Unsloth skips the merge.
    base_model = FastLanguageModel.get_peft_model(
        base_model,
        r=LORA_RANK,
        target_modules=list(LORA_TARGET_MODULES),
        lora_alpha=LORA_ALPHA,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
        max_seq_length=MAX_SEQ_LENGTH,
    )
    export_gguf(base_model, base_tokenizer, gguf_name)


if BASE_ONLY:
    _need(
        DRIVE.is_dir(),
        f"Google Drive has no folder {DRIVE_FOLDER}. Make a folder named grc-training in My Drive (nothing to upload "
        "for the base file), then Runtime -> Run all.",
    )
    GGUF_NAME = f"qwen3.5-4b-base.{QUANT.upper()}.gguf"
else:
    try:
        _identity = json.loads((ADAPTER_DIR / "run.json").read_text())
    except (OSError, ValueError):
        _identity = None
    _need(
        isinstance(_identity, dict) and (ADAPTER_DIR / "adapter_model.safetensors").is_file(),
        f"There is no finished training in {ADAPTER_DIR}. Choose Runtime -> Run all to train first.",
    )
    _tag = run_tag(_identity)
    GGUF_NAME = f"qwen3.5-4b-grc-{_tag}.{QUANT.upper()}.gguf"
_target = DRIVE / GGUF_NAME

if _target.is_file():
    print(f"Already exported: {GGUF_NAME}, {_target.stat().st_size / 2**30:.2f} GB, in {DRIVE_FOLDER}.")
elif BASE_ONLY:
    export_base(GGUF_NAME)
else:
    if globals().get("TRAINED_TAG") == _tag:
        print("Exporting the trained model from this session.")
        _export_model, _export_tokenizer = model, tokenizer
    else:
        print(f"Loading the saved adapter from {ADAPTER_DIR}.")
        for _name in ("trainer", "model"):
            globals().pop(_name, None)
        gc.collect()
        torch.cuda.empty_cache()
        _export_model, _export_processor = FastLanguageModel.from_pretrained(
            model_name=str(ADAPTER_DIR),
            max_seq_length=MAX_SEQ_LENGTH,
            load_in_4bit=LOAD_IN_4BIT,
        )
        _export_tokenizer = getattr(_export_processor, "tokenizer", _export_processor)
    globals().pop("trainer", None)  # the optimiser's memory isn't needed any more
    gc.collect()
    torch.cuda.empty_cache()
    export_gguf(_export_model, _export_tokenizer, GGUF_NAME)

## 10. Next steps (on the Mac)

1. **Download** the `.gguf` file named above from Drive's `grc-training` into `~/models/grc/` (never into the repo).
2. **Load it** in the Unsloth desktop app (port 8888, HexStrike off): context at least 8,192, Parallel Slots 1.
3. **Point the app at it:** set the `AI_TEXT_*` lines in `.env` (the README shows them), then rebuild grc-api and grc-worker.
4. **Run the after-test** on Company X, the same way as the before-test.

In base-only mode the file is the before-test's base file. Set `BASE_ONLY` back to `False` to train.